# Week 10 · Verify, hand over, present

[![Open in Colab](../assets/badges/colab-badge.svg)](https://colab.research.google.com/github/loem-ms/programming-for-information-systems-site/blob/main/content/downloads/Week_10.ipynb)

A system is finished when someone else can trust it. This week you prove that yours works, write the note that lets someone run it without you, and show it in three minutes.

| | |
|---|---|
| **This week's question** | How do you show that your system works, and how can someone else run it without you? |
| **Python you will learn** | tests that can fail (`test_…()` with `assert`, reviewed from Week 4), test functions in a file, sanity checks on numbers, a clean run, reading code written by AI |
| **Principle 10** | **Verify before you trust.** Tests, clean runs and code review, especially for code written by AI. |
| **File you add** | `my_foodbridge/tests.py` → your own `test_…()` functions, and a `README.md` for the next person |
| **What you will see** | `bridge.check("all")`: every part, your tests and your decisions · `bridge.show("gallery")`: all your pictures on one page |

*Campus FoodBridge is fictional. All organisations, people and numbers are invented.*

![System map, week 10: tests.py checks it all; hand over and present](../assets/diagrams/system_map_w10.svg)

### The story so far

Your folder holds every part, and `app.py` runs as a menu. Next month the coordinator hands your system to a new team of volunteers. They cannot ask you questions. The coordinator asks: *"How do I know it works? How do you?"* "I ran it once and saw no error" is not an answer: last week, the AI's Version A ran without an error too. This week you prove it, hand it over, and present it.

## Before you start

1. **File → Save a copy in Drive.** Work in your copy.
2. Run the setup cell. In Colab it asks to connect Google Drive: say yes, so you work in the same `my_foodbridge/` folder as last week.
3. ⭐ = everyone · ⭐⭐ = stretch.

In [ ]:
# ▶ Run this cell first (setup)
# In Colab or Jupyter, this cell unpacks the FoodBridge kit next to the notebook.
# This web page is for reading. To run the notebook, use the 'Open in Colab' button at the top of this page.


### 📁 Your folder first

Your tests will import your files. As in Week 9, `fill_gaps=True` copies the kit's model version of any **missing** Week 4–8 file into your folder, marked 📦. It never touches a file you wrote. (If your Week 9 `app.py` is missing, run the `app.py` cell in your Week 9 notebook again.) Then `check("all")` lists your decisions: count your **🧭** lines. Fewer than three? Tell your instructor now.

In [ ]:
bridge.show(fill_gaps=True)
bridge.check("all", folder="my_foodbridge")

## Session 1 · Learn and try: test your own parts (90 min)

### The idea: verification

Every morning, before the first bus leaves the station, a mechanic walks around it with a checklist: brakes, lights, tyres, doors. She ticks each box and signs the sheet. "It worked yesterday" is not enough, because people's safety depends on the bus.

A system that people depend on needs the same three things: a **checklist anyone can run** (tests), a **check from a clean start** (a clean run), and a **second pair of eyes** (code review). Code written by AI needs them most: it often looks right, and it may say "I tested it" when nobody did.

### 1 · Review quiz: tests from Week 4 (10 min)

Answer on paper first. Then run the cell below to check question 4.

1. `assert 5 > 3` and `assert 3 > 5, "too small"`: what happens with each?
2. A test function's name starts with `____`. It passes when ____.
3. Normal, boundary or odd? `urgency(20)` · `urgency(3)` · `total_portions([])`
4. The rule says *3 hours or less is urgent*. A bug writes `<` instead of `<=`. Which test catches it: `urgency(1) == "urgent"` or `urgency(3) == "urgent"`?

<details><summary>Answers 1–3</summary>

1. The first does nothing (it is true). The second stops with `AssertionError: too small`.
2. `test_`. It passes when no `assert` stops it.
3. Normal (an everyday value) · boundary (the edge of the rule) · odd (an empty list).

</details>

A test is only useful if it **can fail**: on a version with a bug. To see this, we hand a test the function to test as its **input** (as you gave `sorted` a key function in Week 8). `verdict` runs one test on one version and returns `"pass"` or `"fail"`; a crash counts as `"fail"`.

🤔 **Predict** the four results.

In [ ]:
def verdict(test, version):
    """Run one test on one version of a function. Return "pass" or "fail"."""
    try:
        test(version)
    except Exception:            # an assert failed, or the code crashed
        return "fail"
    return "pass"


def urgency_correct(hours_left):
    if hours_left <= 3:
        return "urgent"
    if hours_left <= 12:
        return "soon"
    return "later"


def urgency_bug_3(hours_left):                # 3 hours is no longer urgent
    if hours_left < 3:
        return "urgent"
    if hours_left <= 12:
        return "soon"
    return "later"


def test_one_hour(urgency):
    assert urgency(1) == "urgent"


def test_three_hours(urgency):
    assert urgency(3) == "urgent"


print("1 hour: ", verdict(test_one_hour, urgency_correct), "·", verdict(test_one_hour, urgency_bug_3))
print("3 hours:", verdict(test_three_hours, urgency_correct), "·", verdict(test_three_hours, urgency_bug_3))

**Prediction:** …  **Result:** …

The 1-hour test passes on the buggy version too: it never tries the edge. Inside a test, `urgency` means *the version we hand in*.

### 2 · The expected answer comes from the contract

Where does the number after `==` come from? From the **contract**: the ticket, the rule, your decision. Never from running the code and copying what it printed. If the code is wrong, a copied answer makes the test protect the bug. An AI assistant that writes tests this way will report "all tests pass" for broken code.

🤔 This draft has a bug. The contract says 42 portions are still open in the sample. **Predict** what it prints.

In [ ]:
def open_portions_draft(records):        # a draft: it counts every offer
    total = 0
    for record in records:
        total = total + record["portions"]
    return total


print(open_portions_draft(bridge.sample_records()))     # 🤔

**Prediction:** …  **Result:** …

A test written from this output, `assert open_portions(records) == 100`, would pass and protect the bug. The ticket says 42.

Now your own rules. The versions below are for Try it 1 and 2: one correct version of each rule, and versions with a bug. You do not need to read them before you write your test.

In [ ]:
def label_correct(portions, approved):
    if not approved:
        return "hold"
    if portions >= 20:
        return "team"
    return "standard"


def label_bug_approval(portions, approved):   # forgets approval
    if portions >= 20:
        return "team"
    return "standard"


def label_bug_edge(portions, approved):       # > instead of >=
    if not approved:
        return "hold"
    if portions > 20:
        return "team"
    return "standard"


def urgency_bug_12(hours_left):               # 12 hours is no longer soon
    if hours_left <= 3:
        return "urgent"
    if hours_left < 12:
        return "soon"
    return "later"

### ✍️ Try it 1 · `test_label` (FoodBridge)

The contract from Week 2, with the default team minimum of 20: not approved → `"hold"`; 20 portions or more → `"team"`; otherwise `"standard"`. Write a test that passes on `label_correct` and fails on both buggy versions.

In [ ]:
def test_label(pickup_label):
    # TODO: the edge (19 and 20 portions, approved) and one offer that is not approved
    pass


def try_test_label(version):
    return verdict(test_label, version)


bridge.check("test_label", try_test_label, examples=[
    (label_correct, "pass"),
    (label_bug_approval, "fail"),
    (label_bug_edge, "fail"),
])

### ✍️ Try it 2 · `test_urgency` (FoodBridge)

`urgency` has **two** edges: 3 hours (urgent or soon) and 12 hours (soon or later). Test both sides of each edge.

In [ ]:
def test_urgency(urgency):
    # TODO: both sides of the edge at 3 hours, and both sides of the edge at 12 hours
    pass


def try_test_urgency(version):
    return verdict(test_urgency, version)


bridge.check("test_urgency", try_test_urgency, examples=[
    (urgency_correct, "pass"),
    (urgency_bug_3, "fail"),
    (urgency_bug_12, "fail"),
])

### ⭐⭐ 3 · Check what must not change

Some promises are about what a function must **not** do. In Week 8, `most_urgent_first` had to return a **new** list and leave yours as it was. `.sort()` changes the list itself; `sorted()` makes a new one. A test can check this: call the function, then assert that your list is unchanged. (`[:3]` below keeps the first three items of a list.)

🤔 **Predict** the last line.

In [ ]:
def top_correct(marks):
    return sorted(marks, reverse=True)[:3]


def top_bug_in_place(marks):         # sorts the list it was given
    marks.sort(reverse=True)
    return marks[:3]


def top_bug_lowest(marks):           # the three lowest marks
    return sorted(marks)[:3]


my_marks = [5, 9, 7, 10]
print(top_bug_in_place(my_marks))
print(my_marks)                      # 🤔

**Prediction:** …  **Result:** …

### ⭐⭐ Try it 3 · `test_top_three`

`top_three(marks)` returns the three highest marks, highest first, and leaves the list it was given unchanged. Write a test that passes on `top_correct` and fails on both buggy versions.

In [ ]:
def test_top_three(top_three):
    marks = [5, 9, 7, 10]
    # TODO: assert the answer, then assert that marks is still [5, 9, 7, 10]
    pass


def try_test_top_three(version):
    return verdict(test_top_three, version)


bridge.check("test_top_three", try_test_top_three, examples=[
    (top_correct, "pass"),
    (top_bug_in_place, "fail"),
    (top_bug_lowest, "fail"),
])

### 🎫 This week's ticket

> **From:** the coordinator · **To:** you
> Next month a new volunteer team takes over your system. Before I let them use it, I want proof that anyone can run: tests for the rules that protect people, kept with the system. And a short note, so they can run it without calling you.

| | |
|---|---|
| **File** | `my_foodbridge/tests.py`: `test_…()` functions with `assert`, no inputs |
| **At least** | one test each for the team rule's edge, the urgency edges, an empty list, and a bad row at the door |
| **Uses** | your own files: `rules.py`, `records.py`, `data_model.py` |
| **Done when** | `bridge.check("all", folder="my_foodbridge")` shows ✅ for each test, a clean run works (Lab 3), and `README.md` is in your folder (Session 2) |

| Test | Promise it protects |
|---|---|
| `test_team_minimum_boundary` | 19 portions → `"standard"`, 20 → `"team"`, 20 not approved → `"hold"` |
| `test_urgency_boundaries` | 3 → `"urgent"`, 4 → `"soon"`, 12 → `"soon"`, 13 → `"later"` |
| `test_totals_of_an_empty_list` | `total_portions([])` and `open_portions([])` are both 0 |
| `test_door_refuses_a_word_for_portions` | portions `"many"` → `(None, [one message])` |

### ✏️ Model it on paper first (5 min)

Make a **test plan** before you write code: a table with one row per part you wrote (`pickup_label`, `urgency`, `open_portions`, `clean_offer`, …) and three columns: **normal**, **boundary**, **odd**. Write one input and its expected answer in each box, taken from the contract, not from running the code. Mark the boxes that test one of **your** decisions.

📝 **My test plan:** … (describe it, or paste a photo)

### 🧩 Ladder 1 · Put the lines in order

In [ ]:
bridge.parsons("urgent_first_test")

### 📁 Write your tests file

In the notebook, the function came in as the test's input. In `tests.py` it comes from your own files with `import`, so a test has **no input**: `def test_urgency_boundaries():`. The body stays the same. Your tests then live **with** the system, and anyone can run them again after every change.

The numbers below are the **default** decisions. If you changed a setting (for example `TEAM_PICKUP_MIN_PORTIONS = 25`, `URGENT_HOURS = 4`, or `CATEGORIES`), use **your** values: a test pins your decision. Always edit `tests.py` (and `README.md` in Session 2) in their notebook cells, **not** in the Files panel, so a clean run does not overwrite your edits.

In [ ]:
%%writefile my_foodbridge/tests.py
# tests.py · Week 10: my own tests
from records import open_portions, total_portions
from rules import pickup_label, urgency
from data_model import clean_offer


def test_team_minimum_boundary():
    assert pickup_label(19, True) == "standard"
    # TODO: exactly the team minimum (20) needs a team; 20 portions that are not approved is "hold"


def test_urgency_boundaries():
    assert urgency(3) == "urgent"
    # TODO: 4 is "soon"; 12 (SOON_HOURS) is "soon"; 13 (one hour beyond SOON_HOURS) is "later"


def test_totals_of_an_empty_list():
    # TODO: total_portions([]) and open_portions([]) are both 0
    pass


def test_door_refuses_a_word_for_portions():
    # TODO: a row with "donor", "contact", "food", a valid "category", "portions": "many", "hours_left": "5"
    #       is refused: offer is None, and there is exactly one error
    pass

`bridge.check("all")` checks every part in your folder, runs every `test_…()` function in `tests.py`, and lists your decisions: the settings in your files that differ from the defaults.

In [ ]:
bridge.check("all", folder="my_foodbridge")

### 🧭 Your tests pin your decisions

Try it once: change one setting in a file (the **Files** panel, as in Week 9), then run the cell below. If a test fails now, it is doing its job: it tells you that a decision changed. Decide which is right, the new setting or the old test, then update the test (in the `%%writefile` cell above) and your decision log together. ⭐⭐ Add one more test of your own to `tests.py` (for example, from your test plan, or the storage puzzle `bridge.parsons("save_then_load_test")`).

In [ ]:
bridge.check("all", folder="my_foodbridge")

### 🔬 The verification lab (25 min)

Three checks that professionals run before they trust a system: are the **numbers** sane, does **one constant** still reach every screen, and does everything run from a **clean start**? Note each result with the date: in Session 2 it goes into the verification log of your `README.md`.

### Lab 1 · Sanity-check the numbers (10 min)

A chart can look fine and still be wrong. Before you trust a number, check what **must** be true about it: the parts add up to the whole, a rate stays between 0 % and 100 %, and an empty group does not crash the code (no division by 0). Your Week 9 `rate` is the first thing to test. The versions below have the bugs these checks catch.

In [ ]:
def rate_correct(part, whole):
    if whole == 0:
        return 0.0
    return round(100 * part / whole, 1)


def rate_bug_zero(part, whole):          # crashes on an empty group
    return round(100 * part / whole, 1)


def rate_bug_fraction(part, whole):      # forgets the 100: a fraction, not a percentage
    if whole == 0:
        return 0.0
    return round(part / whole, 1)


def rate_bug_flipped(part, whole):       # divides the wrong way round: more than 100 %
    if part == 0:
        return 0.0
    return round(100 * whole / part, 1)

### ✍️ Try it 4 · `test_rate`

Write a test that passes on `rate_correct` and fails on all three buggy versions: a normal case (3 of 20), the top edge (20 of 20 is 100 %) and the odd case (0 of 0).

In [ ]:
def test_rate(rate):
    # TODO: a normal case, the top edge and an empty group
    pass


def try_test_rate(version):
    return verdict(test_rate, version)


bridge.check("test_rate", try_test_rate, examples=[
    (rate_correct, "pass"),
    (rate_bug_zero, "fail"),
    (rate_bug_fraction, "fail"),
    (rate_bug_flipped, "fail"),
])

Now the same checks on the real semester. Each `assert` is silent when the number is sane.

🤔 **Predict:** which check would stop if the file reader skipped the last row? Which would stop if your `rate` were `rate_bug_flipped`?

In [ ]:
import csv

with open("foodbridge_kit/data/semester.csv", encoding="utf-8") as file:
    offers = list(csv.DictReader(file))


def count_by(rows, field):                       # from Week 6
    counts = {}
    for row in rows:
        counts[row[field]] = counts.get(row[field], 0) + 1
    return counts


statuses = count_by(offers, "status")
parts_total = 0
for status in statuses:
    parts_total = parts_total + statuses[status]
assert parts_total == len(offers) == 404
print("✅ 1 · The statuses add up to the whole:", parts_total, "of 404 offers")

rejected = []
for offer in offers:
    if offer["status"] == "rejected":
        rejected.append(offer)
rejections = count_by(rejected, "category")
by_category = count_by(offers, "category")
for category in by_category:
    assert 0 <= rate_correct(rejections.get(category, 0), by_category[category]) <= 100
print("✅ 2 · Every rejection rate is between 0 and 100 %")

assert rate_correct(0, 0) == 0.0
print("✅ 3 · An empty group gives 0.0, not a crash")

**Prediction:** …  **Result:** …

### 🐤 Lab 2 · The canary drill (5 min)

At Demo day the instructor asks you for a live change at the booth. Rehearse it with your canary from Week 9:

1. Run the cell below and note what choice 1 says about F004.
2. In the **Files** panel, change `URGENT_HOURS` in `my_foodbridge/rules.py` by one (for example 3 → 4) and save. **Predict**, then run the cell again.
3. Change it back, save, and run the cell once more. Your system must look as it did in step 1.

In [ ]:
bridge.show("app", script=["1", "q"])

**Before:** …  **Prediction:** …  **After:** …  **Back:** …

### 🧼 Lab 3 · A clean run (5 min)

A notebook remembers every cell you ran, even cells you later changed or deleted. So a notebook can work today only because of an old cell that is gone. A **clean run** proves that it does not: choose **Runtime → Restart session and run all**. Python forgets everything, then runs every cell from top to bottom.

Remember: this re-runs the `%%writefile` cells in this notebook (`tests.py` above, `README.md` in Session 2), so keep your edits to those two files inside their notebook cells. Edits to earlier weeks' files (`rules.py`, `report.py`, …) stay safe in the **Files** panel. Do a clean run now, and once more just before you submit. Write the date and the result (✅, or what failed and what you changed) in your verification log.

📝 **My lab notes (5 min)** — *Date:* … · *Sanity checks:* … · *Canary:* … · *Clean run:* …

**Before Session 2:** your instructor announces your **pod** (about four classmates) and your **booth** time for Demo day.

### ⭐⭐ Final project Level 4 · A screen written by AI (optional)

*Homework between Week 9 and Week 10, or for fast finishers in Session 1. Not after the code freeze.*

Level 4 grows your Week 8 page into a file: `screen.py`, with one function `make_page(records)` that returns an HTML page as a string. This time an AI writes it, and you check it: you can read its loop and its `html.escape` because you wrote the same 15 lines yourself. Give the AI a contract, not a wish:

> Write a Python file `screen.py` with one function, `make_page(records)`. It returns an HTML page as one string (a `str`). Show one row per open offer (new, approved or picked_up), most urgent first. Import `urgency` from `rules.py` and `most_urgent_first` from `report.py`, and use them. Put all imports at the top of the file. Do not write any of their numbers again. Use only the Python standard library. Never show the contact field.

If you are **not** doing Level 4, leave the cell below commented out (so a clean run does not create a sample `screen.py`). If you **are** doing Level 4, remove the `# ` from every line to try the small working example, or replace the body with the AI's code (keep `%%writefile my_foodbridge/screen.py` as the first line) and **read it before you run it**.

In [ ]:
# %%writefile my_foodbridge/screen.py
# # screen.py · Final project Level 4 (optional): remove the "# " at the start of each line, or paste the AI's code after %%writefile
# from report import most_urgent_first
# from rules import urgency
#
#
# def make_page(records):
#     """Return an HTML page (a str): the open offers, most urgent first."""
#     rows = ""
#     for record in most_urgent_first(records):
#         if record["status"] in ["new", "approved", "picked_up"]:
#             rows = rows + "<li>" + record["id"] + " · " + record["food"] + " · " + urgency(record["hours_left"]) + "</li>"
#     return "<h3>Open offers</h3><ul>" + rows + "</ul>"

In [ ]:
bridge.show("screen")

**The canary test.** Miners once took a small bird, a canary, into the mine: if the air went bad, the bird showed it first. Your canary is one constant:

1. Look at the screen above and count the offers marked "urgent".
2. Open `rules.py` (the **Files** panel) and check your current `URGENT_HOURS`. If it is below `5`, change it to `10`; if it is `5` or more, change it to `1`. Save.
3. Run the `bridge.show("screen")` cell again. Did the urgent count change (between 0 and 2)? Yes → the screen **uses** your rule ✅. No change → the screen **copied** the rule (for example `if record["hours_left"] <= 3:`): send it back to the AI, or fix it.
4. Change `URGENT_HOURS` back (or keep it, and update your decision log and your tests). Write the canary test in your verification log.

## Session 2 · Review, hand over and submit (45 min) · then Demo day

*Starting Session 2 in a new runtime? Run the setup cell at the top first.*

**Minute 45 is the code freeze.** In these 45 minutes you review an AI's tests, finish your `README.md`, do a last clean run and **submit your zip**. At Demo day you present the folder you submitted.

### ⚖️ Good or risky?

You asked an AI assistant: *"Write tests for my FoodBridge rules."* It answered with **Version A** and said: *"All 3 tests pass ✅."* The cell below gives it three **broken** parts and runs its tests. Read the output.

In [ ]:
# Three broken parts
def pickup_label(portions, approved):     # forgets approval
    if portions >= 20:
        return "team"
    return "standard"


def urgency(hours_left):                  # 3 hours is no longer urgent
    if hours_left < 3:
        return "urgent"
    return "later"


def open_portions(records):               # counts delivered food too
    total = 0
    for record in records:
        total = total + record["portions"]
    return total


# Version A · tests from an AI assistant
def test_pickup_label():
    assert pickup_label(30, True) in ["hold", "team", "standard"]


def test_urgency():
    print("urgency(2) is", urgency(2))


def test_open_portions():
    records = [{"portions": 10, "status": "approved"}]
    assert open_portions(records) == open_portions(records)


test_pickup_label()
test_urgency()
test_open_portions()
print("Version A: all 3 tests passed, and all 3 parts are broken.")

**Version B** is the `tests.py` you just wrote: `test_team_minimum_boundary`, `test_urgency_boundaries`, `test_totals_of_an_empty_list`, …

Use **Principle 10 (verify before you trust)**. For each test in Version A: why can it never catch the bug? Which offer or which person is harmed when the broken part is used? Which line of Version B would catch it?

📝 **My review of Version A:**
1. `test_pickup_label`: …
2. `test_urgency`: …
3. `test_open_portions`: …

<details><summary>Instructor's view</summary>

1. **`test_pickup_label` only checks that the answer is one of three words.** Any rule passes, even one that forgets approval. A new, unapproved offer of 40 portions of fruit, like the Green Market Stall's bananas (F003, which the coordinator later rejected), is labelled "team". A team of volunteers rides to the market for food nobody approved. Version B's `pickup_label(20, False) == "hold"` catches it.
2. **`test_urgency` prints and never asserts.** It cannot fail. With the broken rule, an offer with exactly 3 hours left, a common case for the Campus Canteen's rice meals, shows "later". Volunteers wait, and the rice spoils before the Riverside Community Kitchen gets it. Version B's `urgency(3) == "urgent"` catches it.
3. **`test_open_portions` compares the function with itself.** It always passes. And its one record is approved, so it could not see the bug anyway. With the broken part, the board says 100 portions are still open when only 42 are. The coordinator promises food to Hope Children's Centre that was already delivered or rejected. Version B's empty-list test does not catch this one, so add a test with a delivered record (the ⭐⭐ Ladder 2 test after Submit does).

"All tests pass" means nothing until you know the tests **can fail**. Ask of every test, from an AI or from yourself: *which bug would make this fail?* If the answer is "none", it is not a test.

</details>

### 📝 Your handover note: `README.md`

The next person has your folder and nothing else. A `README.md` in the folder tells them what the system does, how to run it, what you decided, how you verified it, and what it does **not** do. Fill in the cell and run it: it saves the file in your folder. It is also where your decision log and your verification log live for the Final project. Paste in your homework drafts and your lab notes.

In [ ]:
%%writefile my_foodbridge/README.md
# My FoodBridge system

Author: (your name) · Date: (date)
Campus FoodBridge is a fictional case from a course.

## What it does
(2–3 sentences: who uses it, and for what.)

## How to run it
1. (step, and what you should see)
2.
3.

## My decisions (decision log)
| Setting (file) | My value (default) | An alternative | Why I chose it | Evidence (data / week) | Who is affected / the tradeoff |
|---|---|---|---|---|---|
| | | | | | |

## How I verified it (verification log)
| Date | What I checked | How | Result | What I changed |
|---|---|---|---|---|
| | | | | |

## Known limits
-

### 🧼 A last clean run

**Runtime → Restart session and run all**, then run the cell below. Check: every test ✅, your three 🧭 decisions still listed (an old `%%writefile` cell can undo a decision), `README.md` complete. Add the date and the result to your verification log.

In [ ]:
bridge.check("all", folder="my_foodbridge")

### Submit (by minute 45)

1. Make sure the clean run above worked and `README.md` is complete.
2. Make a zip of your folder. In Google Drive: right-click `my_foodbridge` → **Download** (Drive makes the zip). Or run the cell below (it also makes sure `saved/offers.json` is present), then in the **Files** panel right-click `my_foodbridge_submission.zip` → **Download**.
3. Email the zip to `submit.progmis@gmail.com` by **Session 2, minute 45**. The file name and the subject line are announced in class. From now on your code is frozen.

In [ ]:
import json
import os
import shutil

os.makedirs("my_foodbridge/saved", exist_ok=True)
if not os.path.exists("my_foodbridge/saved/offers.json"):
    with open("my_foodbridge/saved/offers.json", "w", encoding="utf-8") as f:
        json.dump(bridge.sample_records(), f, indent=2)

shutil.make_archive("my_foodbridge_submission", "zip", ".", "my_foodbridge")
print("Made my_foodbridge_submission.zip: download it from the Files panel.")

### ⭐⭐ After you submit: one more test

While you wait for your booth, try Ladder 2 (it does not change your folder). The versions below are a correct `open_portions` and three versions with a bug.

In [ ]:
def open_correct(records):
    total = 0
    for record in records:
        if record["status"] in ["new", "approved", "picked_up"]:
            total = total + record["portions"]
    return total


def open_bug_all(records):              # counts delivered food too
    total = 0
    for record in records:
        total = total + record["portions"]
    return total


def open_bug_picked_up(records):        # forgets picked_up
    total = 0
    for record in records:
        if record["status"] in ["new", "approved"]:
            total = total + record["portions"]
    return total


def open_bug_empty(records):            # an empty list gives None instead of 0
    if records == []:
        return None
    return open_correct(records)

### 🧩 Ladder 2 · From a blank cell

`open_portions(records)` adds up the portions of the offers that are still open (new, approved or picked_up). Write a test that passes on `open_correct` and fails on all three buggy versions. Hint: make three small records yourself (one approved, one delivered, one picked up) and work out the answer from the contract, then add the odd case.

In [ ]:
def test_open(open_portions):
    pass


def try_test_open(version):
    return verdict(test_open, version)


bridge.check("test_open", try_test_open, examples=[
    (open_correct, "pass"),
    (open_bug_all, "fail"),
    (open_bug_picked_up, "fail"),
    (open_bug_empty, "fail"),
])

## Demo day and final reflection (Session 2, second half · Session 3)

Two things happen at the same time. In your **pod**, everyone presents for 3 minutes (two rounds, with new listeners in Session 3). At the **booth**, the instructor calls you for 3 minutes on your own to explain one decision and make one live change. When the instructor calls you, go; your pod carries on.

*Starting Session 3 in a new runtime? Run the setup cell at the top first.*

### 👀 All your pictures on one page

`bridge.show("gallery")` puts every view of your system on one page: the map, the sheet, the lifecycle, the history, the board, and your app. Use it in your pod demo. You need no slides.

In [ ]:
bridge.show("gallery")

### 📝 Your 3-minute pod demo

You present to your pod (about four classmates): 3 minutes, then 1 minute of questions. Your listeners fill in a peer card for you; you fill one in for each of them.

| Part | Time | What to say and show | My notes |
|---|---|---|---|
| 1 · The people and the problem | 0:30 | Who uses your system? What goes wrong for them without it? | |
| 2 · My decisions | 0:45 | Two or three settings: the value, the alternative, who is affected. One came from the data. | |
| 3 · Live demo | 0:45 | `bridge.show("gallery")` or your menu, plus one chart from your Week 9 analysis (the web app is optional). Show one thing your decisions changed. | |
| 4 · How I verified it | 0:30 | One test and the bug it catches; your clean run; one thing you found in AI code. | |
| 5 · One limit | 0:30 | What your system does not do, and the next step. | |

Three minutes is short. Practise once with a timer. Show one thing well rather than everything quickly.

### 🎤 The booth: one decision and one live change (3 minutes)

1. **0:00–0:45 · Show and explain.** Show one output that your decisions changed (`show("app")`, `show("sheet")` or `show("board")`), and explain one row of your decision log in your own words: the value, the alternative, the evidence, who is affected.
2. **0:45–2:30 · The live change.** The instructor picks a card with a setting you did **not** change. You (1) say the file and the setting, (2) **predict** what will change and what will not, (3) edit it in the **Files** panel, save, and run the cell below, (4) explain any difference from your prediction.
3. **2:30–3:00 · Change it back** while the instructor writes the marks.

Two example cards: *"hide the category from the public"* (`PUBLIC_FIELDS` in `report.py`: the category column leaves the public board) and *"urgent means one hour more than now"* (`URGENT_HOURS` in `rules.py`: F004 turns urgent, and the test that pins your value fails. Why is that good news?).

**Before you go to the booth:** the setup cell has run · the **Files** panel is open at `my_foodbridge/` · the cell below is ready. Your submitted zip is safe, so a live change cannot spoil it; change the setting back after the booth.

In [ ]:
bridge.check("all", folder="my_foodbridge")
bridge.show("app", script=["1", "3", "q"])

**My change:** …  **Prediction:** …  **Result:** …  **Which test or picture showed it:** …

📝 **Peer cards** (one strength and one question for each classmate in your pod): …

## Final reflection

Write alone first (Session 3), then talk in pairs. Submit your answers in the Week 10 part of the [weekly reflection form](https://docs.google.com/forms/d/e/1FAIpQLSdM6P3ZrTaTwMAEcwZNa6-4Wl-ebus3ClRLIZinsTr4kXOBaA/viewform). They count for participation, not for the Final marks.

1. **To your Week 1 self:** in Week 1, how did you answer "What is an information system?" How would you answer in one sentence now? Name one principle that changed your mind.
2. **The decision you are least sure about:** which of your three decisions is it? What data, or whose voice, would change your mind?
3. **A moment you checked before you trusted:** AI code, your own code or a chart that looked right but was wrong. How did you notice? What will you check first next time?
4. **Your next step:** choose one system around you (course registration, a banking app, a ride app). In two lines: who is affected by it, and how would you check that it works?

📝 **My answers:** 1 … · 2 … · 3 … · 4 …

## Finish

- [ ] My tests pass on correct code and fail on buggy code (Try it 1, 2 and 4), and my numbers passed the sanity checks.
- [ ] `tests.py` is in my folder, and `bridge.check("all")` shows ✅ for my tests.
- [ ] I found the problems in the AI's tests and named who would be harmed.
- [ ] My `README.md` has my decision log (6 columns) and my verification log (5 columns), a clean run works, and I submitted my zip by minute 45.
- [ ] I gave my pod demo, and at the booth I explained one decision and made one live change.
- [ ] I wrote my final reflection.

**What you can now do:** read code and judge it, write tests that can fail, run a clean check, and explain every decision in a system you built yourself. This is the last week. Keep asking two questions about every system you meet, including the ones an AI writes for you: *who is affected?* and *how do I know it works?*

⬅ **Previous:** [Week 9 · Assemble and analyse](Week_09.ipynb)